# Surrogate Factory — UCFatigue
## Chapter 5. Feature Selection
Objectives:
- Analyse correlations between inputs and fatigue outputs.
- Add SSE (FEM element number) as an explicit numeric feature.
- Select inputs: `FLAP`, `Altitude`, `TAS`, `Mass`, `q`, `gamma`, `Type_segment`, `Xcg(%CMA)`, `SSE`.
- Fit one StandardScaler + OneHotEncoder per output.

### 0. Workflow initialisation

In [1]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()

2026-06-25 12:24:27 - SurrogateFactoryLogs - INFO - Initializing Workflow...


2026-06-25 12:24:27 - SurrogateFactoryLogs - INFO - Setting up Workflow folders and paths...


2026-06-25 12:24:27 - SurrogateFactoryLogs - INFO - Changing working directory to Data folder: /Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data


2026-06-25 12:24:27 - SurrogateFactoryLogs - INFO - Loading default metadata schema...


2026-06-25 12:24:27 - SurrogateFactoryLogs - INFO - Tracker requested: 'mlflow'


...

Setting tracking uri: file:///Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/mlruns
2026-06-25 12:24:27 - SurrogateFactoryLogs - INFO - Adding methods to Catalog from configuration.


2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Workflow initialization completed successfully.


2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Resuming workflow job: 'UCFATIGUE_1'


2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Workflow data loaded successfully.


2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Workflow metadata successfully loaded for resume.


### 5. Feature Selection

In [2]:
workflow.import_metadata(stage_name="SF_5_Feature_Selection")

2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Importing metadata for stage: 'SF_5_Feature_Selection'


2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Updating metadata for stage 'Feature_Selection' in memory.


In [3]:
Train_set = workflow.load_data(workflow.config['job_name'] + '_Train_set.csv')
Train_set.describe()

2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Loading data from '/Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/UCFATIGUE_1_Train_set.csv' (Format: csv)


2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Successfully loaded data shape: (703, 29)


,SSE,Seg,FLAP,Altitude,TAS,Time,Distance,Pressure,THRUST,Mass,...,1g,Vertical maneuver,Vertical gust,Turn,Frontal gust,n0,Giro,Xcg(%CMA),q,gamma
count,703.0,703.000000,703.000000,703.000000,703.000000,703.000000,703.000000,294.000000,372.000000,703.000000,...,703.000000,703.000000,703.000000,703.000000,703.000000,703.000000,703.000000,703.000000,703.000000,703.000000
mean,2110017.0,41.958748,2.007112,9407.560455,191.474632,6.090768,20.577383,3.207109,7.196075,18849.749644,...,53.619724,19.151555,0.650916,6.364863,0.440913,1.939189,2.012889,19.307297,4520.728548,-0.251953
std,0.0,28.849442,5.107249,7707.632337,48.353307,27.093480,100.071259,1.897508,6.827872,1348.103241,...,8.308015,2.443396,0.167010,0.631141,1.111938,0.079851,1.573626,4.069819,1816.218384,4.665034
min,2110017.0,7.000000,0.000000,18.000000,101.000000,0.100000,0.200000,0.130000,-3.100000,14125.000000,...,31.561000,12.031000,0.346600,4.122000,0.000000,1.730000,0.670000,14.600000,1652.386607,-39.450765
25%,2110017.0,17.000000,0.000000,2000.000000,146.000000,0.500000,1.600000,1.430000,-0.060000,17730.000000,...,47.292000,17.171500,0.509700,6.016500,0.000000,1.890000,1.289500,16.415000,2887.206589,-4.704237
50%,2110017.0,35.000000,0.000000,8000.000000,198.000000,0.940000,2.500000,3.365000,7.330000,18799.000000,...,52.061000,18.893000,0.586700,6.383000,0.000000,1.940000,1.576000,18.500000,3741.014209,0.000000
75%,2110017.0,59.000000,0.000000,15000.000000,230.000000,3.505000,10.500000,4.980000,12.045000,19991.000000,...,61.585500,21.537000,0.806000,6.834000,0.000000,1.990000,1.874000,20.580000,6376.369370,3.038977
max,2110017.0,122.000000,23.000000,26000.000000,297.000000,623.000000,2325.000000,5.580000,23.470000,21886.000000,...,73.308000,24.471000,1.092000,7.530000,3.722000,2.190000,7.599000,29.300000,9637.860375,18.219357


#### 5.1 Analyse correlations

In [4]:
from feature_selection.analyze_correlations import explore_data_analysis
explore_data_analysis(workflow, Train_set)

2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'explore_data_analysis'


Profiling not available.


#### 5.2 Add/select features (including SSE as FEM element ID)

In [5]:
from feature_selection.add_remove_features import add_features
Dataset_features = add_features(workflow, Train_set)
print(f"Selected columns: {Dataset_features.columns.tolist()}")
Dataset_features.head()

2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'add_features'


Selected columns: ['FLAP', 'Altitude', 'TAS', 'Mass', 'q', 'gamma', 'Type_segment', 'Xcg(%CMA)', '1g', 'Vertical maneuver', 'Vertical gust', 'Turn', 'Frontal gust', 'n0', 'Giro']


,FLAP,Altitude,TAS,Mass,q,gamma,Type_segment,Xcg(%CMA),1g,Vertical maneuver,Vertical gust,Turn,Frontal gust,n0,Giro
0,0,4000,146.000000,19749,3068.646862,5.529634,FLT-1,29.3,48.798,18.743,0.5280,6.810,0.0,1.96,1.375
1,0,18500,264.000000,19749,6331.916153,-5.224181,FLT-3,29.3,63.969,21.712,0.7900,6.681,0.0,1.94,1.890
2,0,8500,226.000000,19749,6407.631479,-5.872891,FLT-3,29.3,64.203,21.924,0.8290,6.757,0.0,1.87,2.017
3,0,4500,148.614179,18799,3131.960347,6.475510,FLT-1,16.9,45.826,19.115,0.5933,7.216,0.0,1.82,7.311
4,0,15000,174.000000,19991,3088.142193,2.771270,FLT-1,29.3,49.561,18.959,0.5320,6.872,0.0,1.96,1.397


#### 5.3 Normalisation / preprocessing

In [6]:
from feature_selection.preprocess import preprocessor
preprocessor(workflow, Dataset_features)

2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'preprocessor'


Output: ['1g', 'Vertical maneuver', 'Vertical gust', 'Turn', 'Frontal gust', 'n0', 'Giro']
Number of Categorical Variables: 2 ['FLAP', 'Type_segment']
Number of Numerical Variables: 6 ['Altitude', 'TAS', 'Mass', 'q', 'gamma', 'Xcg(%CMA)']
Saved preprocessor → preprocess_UCFATIGUE_1.pkl
Feature names out: ['num__Altitude', 'num__TAS', 'num__Mass', 'num__q', 'num__gamma', 'num__Xcg(%CMA)', 'cat__FLAP_0', 'cat__FLAP_10', 'cat__FLAP_15', 'cat__FLAP_23', 'cat__Type_segment_FLT-1', 'cat__Type_segment_FLT-10', 'cat__Type_segment_FLT-11', 'cat__Type_segment_FLT-2', 'cat__Type_segment_FLT-3', 'cat__Type_segment_FLT-4', 'cat__Type_segment_FLT-7', 'cat__Type_segment_FLT-8', 'cat__Type_segment_FLT-9']


In [7]:
from feature_selection.postprocess import postprocessor
postprocessor(workflow, Dataset_features)

2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - ▶ Executing Workflow Step: 'postprocessor'


No postprocess method configured.


### Save

In [8]:
workflow.save_data(Dataset_features, workflow.config['job_name'] + '_Dataset_features.csv')
workflow.save_metadata()

2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - File '/Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/UCFATIGUE_1_Dataset_features.csv' already exists — skipping save (delete data/ folder to rerun from scratch)


2026-06-25 12:24:28 - SurrogateFactoryLogs - INFO - Successfully saved workflow metadata to: /Users/martaarnabatmartin/Desktop/Pipelines/UCFatigue/pipeline/data/artifacts/metadata_UCFATIGUE_1.json
